# OpenOBB: train trên dataset riêng

Chọn GPU (Runtime → Change runtime type → G4 / A100 / L4), sửa `ZIP` ở ô 2 rồi chạy lần lượt.
Dataset là file zip định dạng YOLO-OBB **có sẵn `data.yaml`**; notebook tự tìm file đó, không cần viết lại.
Dòng `path:` trong yaml trỏ tới máy khác cũng không sao: thư mục ảnh được tìm lại tính từ vị trí file yaml.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q --force-reinstall --no-deps git+https://github.com/Nguyenchitrai62/OpenOBB.git

In [ ]:
import glob, os

ZIP = "/content/drive/MyDrive/Dataset/tu_ph/Door_Foot/data_train.zip"       # file zip dataset trên Drive
# ZIP = "/content/drive/MyDrive/Dataset/tu_ph/Wall_Color/data/dataset_wall_color_v3.zip"
DATA_DIR = "/content/dataset"                                              # giải nén vào thư mục riêng

!rm -rf {DATA_DIR} && mkdir -p {DATA_DIR} && unzip -q -o "{ZIP}" -d {DATA_DIR}

# tìm data.yaml có sẵn trong dataset (ưu tiên data.yaml / dataset.yaml, nông nhất)
ymls = sorted(glob.glob(f"{DATA_DIR}/**/*.yaml", recursive=True) + glob.glob(f"{DATA_DIR}/**/*.yml", recursive=True),
              key=lambda p: (os.path.basename(p) not in ("data.yaml", "dataset.yaml"), p.count("/")))
DATA = ymls[0] if ymls else DATA_DIR        # không có yaml: dùng thư mục (images/train, images/val, ...)
print("data =", DATA)
if ymls:
    print(open(DATA, encoding="utf-8").read())

In [ ]:
!openobb train \ndata={DATA} \nmodel=vrdet5x \nepochs=100 \nimgsz=1280 \nbatch=16 \nname=vrdet5x

# fine-tune từ checkpoint có sẵn: thêm  weights=/content/drive/MyDrive/.../best.pt
# kết quả: runs/obb/<name>/weights/best.pt  (thêm project=/content/drive/MyDrive/... để lưu thẳng vào Drive)

In [ ]:
# đánh giá lại một checkpoint (thước YOLO + thước DOTA + lỗi nhầm class), không train
# !openobb val model=/content/drive/MyDrive/.../best.pt data={DATA}